**LOADING DATABASE & IMPORTING LIBRARIES**

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from google.colab import drive
drive.mount('/content/drive')

df_train = pd.read_csv('/content/drive/MyDrive/PD/application_train.csv')

print(f'Rows: {df_train.shape[0]}, Columns: {df_train.shape[1]}')

Mounted at /content/drive
Rows: 307511, Columns: 122


In [2]:
target_dist = df_train['TARGET'].value_counts(normalize=True) * 100

print('Percentage of customers up to date with payments (0): {:.2f}%'.format(target_dist[0]))
print('Percentage of customers in default (1): {:.2f}%'.format(target_dist[1]))


Percentage of customers up to date with payments (0): 91.93%
Percentage of customers in default (1): 8.07%


**CORRECTING YEARS VARIABLE**


In [3]:
df_train['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)

print('Corrected Years')
print(f'Max Days Value: {df_train["DAYS_EMPLOYED"].max()}')


Corrected Years
Max Days Value: 0.0


**TRANSLATE TIME VARIABLES FOR THE BUSINESS**


In [4]:
df_train['AGE_YEARS'] = df_train['DAYS_BIRTH'] / -365
df_train['EMPLOYMENT_YEARS'] = df_train['DAYS_EMPLOYED'] / -365

print(df_train[['AGE_YEARS', 'EMPLOYMENT_YEARS']].describe())

           AGE_YEARS  EMPLOYMENT_YEARS
count  307511.000000     252137.000000
mean       43.936973          6.531971
std        11.956133          6.406466
min        20.517808         -0.000000
25%        34.008219          2.101370
50%        43.150685          4.515068
75%        53.923288          8.698630
max        69.120548         49.073973


**MISSING VALUES**

In [5]:
missing_threshold = 0.60
nulls_per_column = df_train.isnull().mean()
columns_to_drop = nulls_per_column[nulls_per_column > missing_threshold].index

df_train = df_train.drop(columns=columns_to_drop)
print(f'Already Deleted {len(columns_to_drop)} columns due to too many nulls')

num_cols = df_train.select_dtypes(include=['float64','int64',]).columns
cat_cols = df_train.select_dtypes(include=['object',]).columns

df_train[num_cols] = df_train[num_cols].fillna(df_train[num_cols].median())

df_train[cat_cols] = df_train[cat_cols].fillna(df_train[cat_cols].mode().iloc[0])

print('Null Values Successfully Imputed')

Already Deleted 17 columns due to too many nulls
Null Values Successfully Imputed


**CREATION OF FINANCIAL RATIOS**

In [6]:
df_train['CREDIT_INCOME_RATIO'] = df_train['AMT_CREDIT'] / df_train['AMT_INCOME_TOTAL']
df_train['ANNUITY_INCOME_RATIO'] = df_train['AMT_ANNUITY'] / df_train['AMT_INCOME_TOTAL']

print(df_train[['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO']].head())

   CREDIT_INCOME_RATIO  ANNUITY_INCOME_RATIO
0             2.007889              0.121978
1             4.790750              0.132217
2             2.000000              0.100000
3             2.316167              0.219900
4             4.222222              0.179963


**CALCULATING WoE & IV**

In [8]:
def calculate_woe_iv(df, feature, target):
  if pd.api.types.is_numeric_dtype(df[feature]):
    df['binned'] = pd.qcut(df[feature], q=10, duplicates='drop')
  else:
    df['binned'] = df[feature]

  grouped = df.groupby('binned', as_index=False).agg(
      Total=(target, 'count'),
      Bad=(target, 'sum')
  )
  grouped['Good'] = grouped['Total'] - grouped['Bad']

  grouped['Dist_Good'] = np.maximum(grouped['Good'],0.0001) / df[df[target] == 0].shape[0]
  grouped['Dist_Bad'] = np.maximum(grouped['Bad'],0.0001) / df[df[target] == 1].shape[0]

  grouped['WoE'] = np.log(grouped['Dist_Good'] / grouped['Dist_Bad'])

  grouped['IV_bin'] = (grouped['Dist_Good'] - grouped['Dist_Bad']) * grouped['WoE']

  iv_total = grouped['IV_bin'].sum()

  df.drop(columns=['binned'], inplace=True)

  return grouped, iv_total

table_woe, iv_ratio_annuity = calculate_woe_iv(df_train, 'ANNUITY_INCOME_RATIO', 'TARGET')

print(f'The Information Value of ANNUITY_INCOME_RATIO is: {iv_ratio_annuity:.4f}')
print('\nWoE Table by Ranks')
print(table_woe[['binned', 'Total', 'Good', 'Good', 'WoE', 'IV_bin']])

The Information Value of ANNUITY_INCOME_RATIO is: 0.0059

WoE Table by Ranks
              binned  Total   Good   Good       WoE    IV_bin
0  (-0.000776, 0.08]  31647  29389  29389  0.133655  0.001738
1      (0.08, 0.104]  29857  27688  27688  0.114247  0.001208
2     (0.104, 0.125]  30758  28354  28354  0.035149  0.000122
3     (0.125, 0.144]  30751  28339  28339  0.031297  0.000097
4     (0.144, 0.163]  30760  28337  28337  0.026676  0.000070
5     (0.163, 0.186]  30738  28164  28164 -0.039902  0.000162
6     (0.186, 0.212]  30768  28141  28141 -0.061100  0.000383
7     (0.212, 0.247]  30754  28031  28031 -0.100909  0.001062
8     (0.247, 0.302]  30728  28012  28012 -0.099013  0.001021
9     (0.302, 1.876]  30750  28231  28231 -0.015927  0.000026


**SELECTION OF VARIABLES AND DATA SPLITTING**

In [9]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = ['CREDIT_INCOME_RATIO', 'ANNUITY_INCOME_RATIO', 'AGE_YEARS', 'EMPLOYMENT_YEARS', 'AMT_INCOME_TOTAL']

X = df_train[features]
y = df_train['TARGET']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**TRAINING LOGISTIC REGRESSION**

In [10]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(class_weight='balanced', random_state=42)

log_reg.fit(X_train_scaled, y_train)

print('Model Successfully Trained')

Model Successfully Trained


**CALCULATING PROBABILITY OF DEFAULT (PD)**


In [11]:
pd_predicciones = log_reg.predict_proba(X_test_scaled)[:,1]

results = pd.DataFrame({
    'SK_ID_CURR': df_train.loc[y_test.index, 'SK_ID_CURR'],
    'TARGET_REAL': y_test,
    'PD': pd_predicciones
})

results['RISK_DECILE'] = pd.qcut(results['PD'], q=10, labels=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10])

print(results.head())

        SK_ID_CURR  TARGET_REAL        PD RISK_DECILE
199280      331034            0  0.428478           3
122967      242581            0  0.529218           7
156663      281583            0  0.429079           3
134382      255865            0  0.461586           5
250108      389379            0  0.461226           5


**MODEL EVALUATION**  

In [12]:
from sklearn.metrics import roc_auc_score
from scipy.stats import ks_2samp

auc = roc_auc_score(results['TARGET_REAL'], results['PD'])
gini = (2 * auc) - 1

pd_good = results[results['TARGET_REAL'] == 0]['PD']
pd_bad = results[results['TARGET_REAL'] == 1]['PD']

ks_stat, p_value = ks_2samp(pd_bad, pd_good)

print(f'ROC AUC : {auc:.4f} (acceptable in commercial banking > 0.65)')
print(f'Gini Coefficient: {gini:.4f} (Ideal > 0.30)')
print(f'KS Statistic: {ks_stat:.4f} (Ideal > 0.25)')

ROC AUC : 0.6026 (acceptable in commercial banking > 0.65)
Gini Coefficient: 0.2053 (Ideal > 0.30)
KS Statistic: 0.1530 (Ideal > 0.25)


In [ ]:
results.to_csv('predicciones_Pd_home_credit.csv', index=False)
print("File 'predicciones_Pd_home_credit.csv' saved ")

File 'predicciones_Pd_home_credit.csv' saved 
